In [34]:
import pandas as pd
import os

In [24]:
cities = pd.read_csv("../../data/cities.csv")
districts = pd.read_csv("../../data/districts.csv")
provinces = pd.read_csv("../../data/provinces.csv")

### Cities

In [25]:
print(cities.columns.tolist())
cities.head()

['city id', 'district_id', 'name_en', 'name_si', 'name_ta', 'sub_name_en', 'sub_name_si', 'sub_name_ta', 'postcode', 'latitude', 'longitude']


,city id,district_id,name_en,name_si,name_ta,sub_name_en,sub_name_si,sub_name_ta,postcode,latitude,longitude
0,1,1,Akkaraipattu,අක්කරපත්තුව,அக்கரைப்பற்று,NULL,NULL,NULL,32400.0,7.218428,81.854116
1,2,1,Ambagahawatta,අඹගහවත්ත,அம்பகஹவத்த,NULL,NULL,NULL,90326.0,7.301756,81.674729
2,3,1,Ampara,අම්පාර,அம்பாறை,NULL,NULL,NULL,32000.0,7.301756,81.674729
3,4,1,Bakmitiyawa,බක්මිටියාව,பக்மிடியாவ,NULL,NULL,NULL,32024.0,7.029632,81.680205
4,5,1,Deegawapiya,දීඝවාපිය,தீகவாபி,NULL,NULL,NULL,32006.0,7.301756,81.674729


### Districts

In [26]:
print(districts.columns.tolist())
districts.head()

['district id', 'province_id', 'name_en', 'name_si', 'name_ta']


,district id,province_id,name_en,name_si,name_ta
0,1,6,Ampara,අම්පාර,அம்பாறை
1,2,8,Anuradhapura,අනුරාධපුරය,அனுராதபுரம்
2,3,7,Badulla,බදුල්ල,பதுளை
3,4,6,Batticaloa,මඩකලපුව,மட்டக்களப்பு
4,5,1,Colombo,කොළඹ,கொழும்பு


### Provinces

In [27]:
print(provinces.columns.tolist())
provinces.head()

['provinces_id', 'name_en', 'name_si', 'name_ta']


,provinces_id,name_en,name_si,name_ta
0,1,Western,බස්නාහිර,மேல்
1,2,Central,මධ්‍යම,மத்திய
2,3,Southern,දකුණු,தென்
3,4,North Western,වයඹ,வட மேல்
4,5,Sabaragamuwa,සබරගමුව,சபரகமுவ


In [30]:
# Rename inconsistent ID columns
cities = cities.rename(columns={"city id": "city_id"})
districts = districts.rename(columns={"district id": "district_id"})
provinces = provinces.rename(columns={"provinces_id": "provinces_id"})  # already correct, no change needed

# Keep only the relevant columns from each table
# (join keys are kept alongside name_en so we can still merge later)
cities = cities[["city_id", "district_id", "name_en"]]
districts = districts[["district_id", "province_id", "name_en"]]
provinces = provinces[["provinces_id", "name_en"]]

print("CITIES")
display(cities.head())

print("DISTRICTS")
display(districts.head())

print("PROVINCES")
display(provinces.head())

CITIES


,city_id,district_id,name_en
0,1,1,Akkaraipattu
1,2,1,Ambagahawatta
2,3,1,Ampara
3,4,1,Bakmitiyawa
4,5,1,Deegawapiya


DISTRICTS


,district_id,province_id,name_en
0,1,6,Ampara
1,2,8,Anuradhapura
2,3,7,Badulla
3,4,6,Batticaloa
4,5,1,Colombo


PROVINCES


,provinces_id,name_en
0,1,Western
1,2,Central
2,3,Southern
3,4,North Western
4,5,Sabaragamuwa


In [32]:
# Step 1: Rename name_en in each table before merging, so we don't get name_en_x/name_en_y confusion
cities_renamed = cities.rename(columns={"name_en": "city_name"})
districts_renamed = districts.rename(columns={"name_en": "district_name"})
provinces_renamed = provinces.rename(columns={"name_en": "province_name"})

# Step 2: Merge cities with districts on district_id
city_district = cities_renamed.merge(
    districts_renamed,
    on="district_id",
    how="left"
)

# Step 3: Merge the result with provinces
# Note: join keys differ in name (province_id vs provinces_id)
location_hierarchy = city_district.merge(
    provinces_renamed,
    left_on="province_id",
    right_on="provinces_id",
    how="left"
)

# Step 4: Keep only the columns we actually need, in a clean order
location_hierarchy = location_hierarchy[[
    "city_id", "city_name",
    "district_id", "district_name",
    "provinces_id", "province_name"
]]

print(f"Total rows: {len(location_hierarchy)}")
display(location_hierarchy.head(5))

print(f"Total rows: {len(location_hierarchy)}")
display(location_hierarchy.tail(5))

Total rows: 2155


,city_id,city_name,district_id,district_name,provinces_id,province_name
0,1,Akkaraipattu,1,Ampara,6,Eastern
1,2,Ambagahawatta,1,Ampara,6,Eastern
2,3,Ampara,1,Ampara,6,Eastern
3,4,Bakmitiyawa,1,Ampara,6,Eastern
4,5,Deegawapiya,1,Ampara,6,Eastern


Total rows: 2155


,city_id,city_name,district_id,district_name,provinces_id,province_name
2150,2209,Vankalai,15,Mannar,9,Northern
2151,2210,Vellankulam,15,Mannar,9,Northern
2152,2211,Veppankulam,15,Mannar,9,Northern
2153,2212,Vidataltivu,15,Mannar,9,Northern
2154,2213,Mabola,7,Gampaha,1,Western


In [33]:
# Any cities that failed to match a district?
print("Cities with missing district:", location_hierarchy["district_name"].isna().sum())

# Any districts that failed to match a province?
print("Districts with missing province:", location_hierarchy["province_name"].isna().sum())

# Sri Lanka should have 9 provinces and 25 districts — confirm counts look right
print("Unique provinces:", location_hierarchy["province_name"].nunique())
print("Unique districts:", location_hierarchy["district_name"].nunique())
print("Unique cities:", location_hierarchy["city_id"].nunique())

Cities with missing district: 0
Districts with missing province: 0
Unique provinces: 9
Unique districts: 25
Unique cities: 2155


In [37]:
REPO_ROOT = "d:/VanujaS/Git/Exide-Sales-Forecast/"
# Save locally first, inside the notebook's exploration flow
output_path = os.path.join(REPO_ROOT, "data", "location_hierarchy.parquet")
location_hierarchy.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")

Saved to d:/VanujaS/Git/Exide-Sales-Forecast/data\location_hierarchy.parquet


In [38]:
csv_path = os.path.join(REPO_ROOT, "data", "location_hierarchy.csv")
location_hierarchy.to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

Saved to d:/VanujaS/Git/Exide-Sales-Forecast/data\location_hierarchy.csv
